In [1]:
!pip install -q natasha beautifulsoup4

import urllib.request
from bs4 import BeautifulSoup

from natasha import Segmenter, NewsEmbedding, NewsMorphTagger, Doc, MorphVocab

import nltk
from nltk import FreqDist
from nltk.corpus import stopwords

nltk.download("stopwords")

DATA_URL = "http://az.lib.ru/t/tolstoj_a_k/text_0170.shtml"

# Скачиваем html
opener = urllib.request.URLopener({})
resource = opener.open(DATA_URL)
raw_text = resource.read().decode(resource.headers.get_content_charset())

# Удаляем html-теги
soup = BeautifulSoup(raw_text, features="html.parser")

for script in soup(["script", "style"]):
    script.extract()

cleaned_text = soup.get_text()

# Инициализация Natasha
segmenter = Segmenter()
emb = NewsEmbedding()
morph_vocab = MorphVocab()
morph_tagger = NewsMorphTagger(emb)

# Создаем документ
doc = Doc(cleaned_text)

# Сегментация
doc.segment(segmenter)

# Морфологическая разметка
doc.tag_morph(morph_tagger)

# Лемматизация и отбор только буквенных токенов
non_uniq_tokens = []

for token in doc.tokens:
    token.lemmatize(morph_vocab)
    if token.text.isalpha():
        non_uniq_tokens.append(token.lemma)

# Стоп-слова и частоты
STOPWORDS = set(stopwords.words("russian"))
freq_dist = FreqDist(non_uniq_tokens)

# Доля среди 100 самых частотных токенов, не входящих в стоп-лист
top_100_tokens = [token for token, freq in freq_dist.most_common(100)]
share_non_stopwords = sum(token not in STOPWORDS for token in top_100_tokens) / 100

# Сколько токенов встречается строго больше 50 раз
tokens_more_than_50 = sum(freq > 50 for freq in freq_dist.values())

tokens_more_than_10 = sum(freq > 10 for freq in freq_dist.values())

print("Количество предложений:", len(doc.sents))
print("Количество токенов, состоящих только из букв:", len(non_uniq_tokens))
print("Доля не-стоп-слов среди 100 самых частотных токенов:", round(share_non_stopwords, 2))
print("Сколько токенов встречается строго больше 50 раз:", tokens_more_than_50)
print("Сколько токенов встречается строго больше 10 раз:", tokens_more_than_10)

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/afafos/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Количество предложений: 1361
Количество токенов, состоящих только из букв: 20928
Доля не-стоп-слов среди 100 самых частотных токенов: 0.45
Сколько токенов встречается строго больше 50 раз: 55
Сколько токенов встречается строго больше 10 раз: 276
